# Darling Downs Case Studies: Esk (040075) and Millmerran (041069)

This notebook provides a follow-along guide for generating 100+ year hourly rainfall sequences for the Darling Downs region. It implements the methodologies presented in the **HWRS 2025** papers by Millard et al. and Batchelor et al.

## 📖 Research Context
Traditional continuous simulation often struggles with short observed records. The approach used here overcomes this by:
1.  **Splicing Observed and Synthetic Data**: Using historical gauge data where available and filling gaps with stochastically generated or reanalysis data.
2.  **Regionalised Method of Fragments**: Borrowing storm patterns (fragments) from statistically similar donor stations.
3.  **Recursive IFD Fitting**: Ensuring synthetic bursts match the local Intensity-Frequency-Duration characteristics.
4.  **Two-Phase Climate Scaling**: Uplifting extremes for temperature increases (ARR v4.2) while adjusting total volumes for GCM-predicted changes.

---

## Step 1: Site Selection and Data Acquisition

We can choose between two key sites from the Darling Downs Flood Study:
*   **Esk Post Office (040075)**: High-quality records starting from 1886.
*   **Millmerran Post Office (041069)**: 116 years of patched point daily totals used in the *"Of fragments and cubes"* paper.

In [ ]:
from pyraingen.silo import get_silo_point_data, prepare_silo_for_pyraingen
import pandas as pd

# Choose your site
STATION_ID = "040075" # Esk
# STATION_ID = "041069" # Millmerran

EMAIL = "your.email@example.com" 
START = "19800101"
END = "20231231"

print(f"Fetching daily rainfall for Station {STATION_ID} via SILO API...")
silo_df = get_silo_point_data(STATION_ID, START, END, EMAIL)

# Prepare the 1D array for the disaggregation engine
daily_rain = prepare_silo_for_pyraingen(silo_df, 1980, 2023)
print(f"Total volume fetched: {daily_rain.sum():.1f} mm")

## Step 2: Disaggregation (Method of Fragments)

This step takes the daily totals and distributes them into hourly increments by sampling "fragments" from nearby pluviograph stations. 

**Why this works**: By using "state-based" logic, we ensure that if a day was preceded or followed by rain, the chosen fragment reflects that continuity, avoiding the 

In [ ]:
from pyraingen.regionalisedsubdailysim import regionalisedsubdailysim

# Point to the local directory containing pluviograph NetCDF files
PATH_SUBDAILY = "src/pyraingen/data/example/subdaily/" 

print("Running Regionalised Method of Fragments disaggregation...")
regionalisedsubdailysim(
    fnameInput=None, 
    pathSubDaily=PATH_SUBDAILY,
    targetIndex=int(STATION_ID),
    suppliedDailyRain=daily_rain,
    genSeqOption=5,              # Use user-supplied daily array
    nSims=1,
    fnameSubDaily=f"site_{STATION_ID}_hourly_raw.nc"
)

## Step 3: IFD Conditioning (Recursive Rescaling)

As noted in **Batchelor et al. (2025)**, stochastically generated bursts can sometimes underestimate short-duration intensities. We use `ifdcond` to reconcile our hourly sequence with the official BoM 2016 IFD curves.

In [ ]:
from pyraingen.ifdcond import ifdcond

# Path to the target IFD values (CSV from BoM Design Rainfall system)
IFD_CSV = "src/pyraingen/data/example/ifd/targetifds.csv"
DURATIONS = [60, 360, 720, 1440] 
AEPS = [63.2, 50, 20, 10, 5, 2, 1]

ifdcond(
    f"site_{STATION_ID}_hourly_raw.nc", 
    f"site_{STATION_ID}_hourly_conditioned.nc", 
    IFD_CSV,
    nSims=1,
    TargetIFDdurationsEst=DURATIONS,
    TargetIFDdurations=DURATIONS,
    AEP=AEPS,
    plot=True
)

## Step 4: Future Climate Projection (2090 SSP3 Horizon)

Following the **two-phase process** from the HWRS papers:
1.  **IFD Uplift**: Use ARR v4.2 temperature-based scaling for the annual maxima.
2.  **Volume Adjustment**: Use the CMIP6 scaling factor ($\beta$) to adjust the sub-maximal (non-extreme) rainfall.

In [ ]:
from pyraingen.climate import apply_arr_v4_2_uplift, calculate_submaximal_scaling_factor
import numpy as np

# PHASE 1: Intense Event Uplift
DELTA_T = 3.5 # Degrees increase for 2090 SSP3-7.0
ALPHA = 5.0   # 5% increase per degree per ARR v4.2

print("Applying ARR v4.2 IFD Uplift...")
# Assume historical_depths were extracted from the conditioned NetCDF
historical_depths = np.array([[35.0, 45.0, 60.0], [50.0, 70.0, 95.0]]) 
future_depths = apply_arr_v4_2_uplift(historical_depths, ALPHA, DELTA_T)

# PHASE 2: Seasonal Volume Scaling
GCM_SCALE = 0.93 # CMIP6 predicts a 7% reduction in total annual volume

Vo = daily_rain.sum() 
Vm = 15000.0 # Volume of uplifted extremes (example)
Vnm = Vo - 15000.0 # Sub-maximal volume

beta = calculate_submaximal_scaling_factor(Vo, Vm, Vnm, GCM_SCALE)

print(f"\nClimate Projection Parameters:")
print(f"- Temperature Increase: +{DELTA_T} C")
print(f"- Sub-maximal scaling (beta): {beta:.4f}")
print("\nSuccess: You have generated a future-climate ready sequence!")